This script will be used to generate all data needed to train NN model for earhtquake shakemap prediction. Random earthquakes will be simulated and corresponding shakemaps will be generated

Goal of script: Generate M samples x 75 x 75 x 5 Input volume (See channel descriptions below) and M samples x 75 x 75 x 3 "ground truth" shakemap image

Input Volume:
Channe1 1 -> Distance from discrete points to earthquake rupture  
Channel 2 --> Vs30 values for all discrete points  
Channel 3 --> Latitude for all discrete points  
Channel 4 --> Longitude for all discrete points  
Channel 5 --> Magnitude of rupture (single scalar brodcasted to match 75 x 75 grid)

Specify # of training and validation samples to generate at bottom of script

Import Vs30 Raster Downloaded from USGS for Site Soil Properties

In [ ]:
from google.colab import files
uploaded = files.upload()

Initialize All Functions Necessary to Simulate Earthquakes and Run Data Generator

In [ ]:
# Import libraries
!pip install geopandas shapely pyproj rasterio pillow requests -q
import os
import math
import zipfile
import requests
import numpy as np
import geopandas as gpd
from shapely.geometry import Point, LineString
import rasterio
from rasterio.warp import transform as rio_transform
from rasterio.transform import rowcol
import pyproj
from pyproj import Transformer
from PIL import Image
import warnings
warnings.filterwarnings("ignore")

#################  Ground Motion Model Function  ##########################
def gmm_bssa2014(M, Rjb, Vs30, T=0.0, Fault_Type=0, region=1, z1=999):
    """
    Boore, Stewart, Seyhan & Atkinson (2014) NGA-West2 Ground Motion Model
    Inputs:
      - M: scalar magnitude
      - Rjb: scalar or 1D array (km)
      - Vs30: scalar or 1D array (m/s), same shape as Rjb
      - T: spectral period (only T=0 supported here)
      - Fault_Type: 0 = unspecified, 1 = strike strike slip, 2 = Normal, 3 = Reverse
      - region: 0 for California, 1 for Japan, 2 for Italy , 3 else
      - z1: reupture depth [m] -> 999 for unknown
    Returns:
      median shaking intensity [g], sigma of shaking intensity [g]
    """
    R_arr  = np.atleast_1d(np.asarray(Rjb,  dtype=float))
    VS_arr = np.atleast_1d(np.asarray(Vs30, dtype=float))
    if R_arr.shape != VS_arr.shape:
        raise ValueError("Rjb and Vs30 must have the same shape (or be scalars).")

    period = np.array([-1, 0, 0.01, 0.02, 0.03, 0.05, 0.075, 0.1, 0.15, 0.2, 0.25,
                       0.3, 0.4, 0.5, 0.75, 1, 1.5, 2, 3, 4, 5, 7.5, 10], dtype=float)

    U  = 1 if Fault_Type == 0 else 0
    SS = 1 if Fault_Type == 1 else 0
    NS = 1 if Fault_Type == 2 else 0
    RS = 1 if Fault_Type == 3 else 0

    T_arr = np.atleast_1d(np.array(T, dtype=float))
    if T_arr.size != 1:
        raise ValueError("This wrapper supports a single period T.")
    Ti = float(T_arr[0])
    match = np.where(np.abs(period - Ti) < 1e-4)[0]
    if match.size == 0:
        raise ValueError(f"Requested period {Ti} s not in grid.")
    ip = int(match[0])

    med_out = np.empty_like(R_arr, dtype=float)
    sig_out = np.empty_like(R_arr, dtype=float)

    for i in range(R_arr.size):
        med, sig = _BSSA_2014_sub(M, ip, float(R_arr[i]), U, SS, NS, RS, region, z1, float(VS_arr[i]))
        med_out[i] = med
        sig_out[i] = sig

    if np.isscalar(Rjb) and np.isscalar(Vs30):
        return med_out.item(), sig_out.item(), Ti
    else:
        return med_out, sig_out, Ti
#Helper function
def _BSSA_2014_sub(M, ip, Rjb, U, SS, NS, RS, region, z1, Vs30):
    mref = 4.5; rref = 1.0; v_ref = 760.0
    f1 = 0.0; f3 = 0.1; v1 = 225.0; v2 = 300.0
    period = np.array([-1,0,0.01,0.02,0.03,0.05,0.075,0.1,0.15,0.2,0.25,0.3,0.4,0.5,
                       0.75,1,1.5,2,3,4,5,7.5,10], dtype=float)

    mh  = np.array([6.2,5.5,5.5,5.5,5.5,5.5,5.5,5.54,5.74,5.92,6.05,6.14,6.2,6.2,6.2,6.2,6.2,6.2,6.2,6.2,6.2,6.2,6.2])
    e0  = np.array([5.037,0.4473,0.4534,0.48598,0.56916,0.75436,0.96447,1.1268,1.3095,1.3255,1.2766,1.2217,1.1046,0.96991,0.66903,0.3932,-0.14954,-0.58669,-1.1898,-1.6388,-1.966,-2.5865,-3.0702])
    e1  = np.array([5.078,0.4856,0.4916,0.52359,0.6092,0.79905,1.0077,1.1669,1.3481,1.359,1.3017,1.2401,1.1214,0.99106,0.69737,0.4218,-0.11866,-0.55003,-1.142,-1.5748,-1.8882,-2.4874,-2.9537])
    e2  = np.array([4.849,0.2459,0.2519,0.29707,0.40391,0.60652,0.77678,0.8871,1.0648,1.122,1.0828,1.0246,0.89765,0.7615,0.47523,0.207,-0.3138,-0.71466,-1.23,-1.6673,-2.0245,-2.8176,-3.3776])
    e3  = np.array([5.033,0.4539,0.4599,0.48875,0.55783,0.72726,0.9563,1.1454,1.3324,1.3414,1.3052,1.2653,1.1552,1.012,0.69173,0.4124,-0.1437,-0.60658,-1.2664,-1.7516,-2.0928,-2.6854,-3.1726])
    e4  = np.array([1.073,1.431,1.421,1.4331,1.4261,1.3974,1.4174,1.4293,1.2844,1.1349,1.0166,0.95676,0.96766,1.0384,1.2871,1.5004,1.7622,1.9152,2.1323,2.204,2.2299,2.1187,1.8837])
    e5  = np.array([-0.1536,0.05053,0.04932,0.053388,0.061444,0.067357,0.073549,0.055231,-0.042065,-0.11096,-0.16213,-0.1959,-0.22608,-0.23522,-0.21591,-0.18983,-0.1467,-0.11237,-0.04332,-0.014642,-0.014855,-0.081606,-0.15096])
    e6  = np.array([0.2252,-0.1662,-0.1659,-0.16561,-0.1669,-0.18082,-0.19665,-0.19838,-0.18234,-0.15852,-0.12784,-0.092855,-0.023189,0.029119,0.10829,0.17895,0.33896,0.44788,0.62694,0.76303,0.87314,1.0121,1.0651])

    c1  = np.array([-1.24300,-1.13400,-1.13400,-1.13940,-1.14210,-1.11590,-1.08310,-1.06520,-1.05320,-1.06070,-1.07730,-1.09480,-1.12430,-1.14590,-1.17770,-1.19300,-1.20630,-1.21590,-1.21790,-1.21620,-1.21890,-1.25430,-1.32530])
    c2  = np.array([0.14890,0.19170,0.19160,0.18962,0.18842,0.18709,0.18225,0.17203,0.15401,0.14489,0.13925,0.13388,0.12512,0.12015,0.11054,0.10248,0.09645,0.09636,0.09764,0.10218,0.10353,0.12507,0.15183])
    c3  = np.array([-0.00344,-0.00809,-0.00809,-0.00807,-0.00834,-0.00982,-0.01058,-0.01020,-0.00898,-0.00772,-0.00652,-0.00548,-0.00405,-0.00322,-0.00193,-0.00121,-0.00037,0.00000,0.00000,-0.00005,0.00000,0.00000,0.00000])
    h   = np.array([5.3,4.5,4.5,4.5,4.49,4.2,4.04,4.13,4.39,4.61,4.78,4.93,5.16,5.34,5.6,5.74,6.18,6.54,6.93,7.32,7.78,9.48,9.66])

    deltac3_gloCATW = np.array([0.0]*23, dtype=float)
    deltac3_CHTU    = np.array([0.00435,0.00286,0.00282,0.00278,0.00276,0.00296,0.00296,0.00288,0.00279,0.00261,0.00244,0.00220,0.00211,0.00235,0.00269,0.00292,0.00304,0.00292,0.00262,0.00261,0.00260,0.00260,0.00303])
    deltac3_ITJA    = np.array([-0.00033,-0.00255,-0.00244,-0.00234,-0.00217,-0.00199,-0.00216,-0.00244,-0.00271,-0.00297,-0.00314,-0.00330,-0.00321,-0.00291,-0.00253,-0.00209,-0.00152,-0.00117,-0.00119,-0.00108,-0.00057,0.00038,0.00149])

    c    = np.array([-0.8400,-0.6000,-0.6037,-0.5739,-0.5341,-0.4580,-0.4441,-0.4872,-0.5796,-0.6876,-0.7718,-0.8417,-0.9109,-0.9693,-1.0154,-1.0500,-1.0454,-1.0392,-1.0112,-0.9694,-0.9195,-0.7766,-0.6558])
    vc   = np.array([1300.00,1500.00,1500.20,1500.36,1502.95,1501.42,1494.00,1479.12,1442.85,1392.61,1356.21,1308.47,1252.66,1203.91,1147.59,1109.95,1072.39,1009.49,922.43,844.48,793.13,771.01,775.00])
    f4   = np.array([-0.1000,-0.1500,-0.1483,-0.1471,-0.1549,-0.1963,-0.2287,-0.2492,-0.2571,-0.2466,-0.2357,-0.2191,-0.1958,-0.1704,-0.1387,-0.1052,-0.0679,-0.0361,-0.0136,-0.0032,-0.0003,-0.0001,0.0000])
    f5   = np.array([-0.00844,-0.00701,-0.00701,-0.00728,-0.00735,-0.00647,-0.00573,-0.00560,-0.00585,-0.00614,-0.00644,-0.00670,-0.00713,-0.00744,-0.00812,-0.00844,-0.00771,-0.00479,-0.00183,-0.00152,-0.00144,-0.00137,-0.00136])
    f6   = np.array([-9.9,-9.9,-9.9,-9.9,-9.9,-9.9,-9.9,-9.9,-9.9,-9.9,-9.9,-9.9,0.092,0.367,0.638,0.871,1.135,1.271,1.329,1.329,1.183])
    f7   = np.array([-9.9,-9.9,-9.9,-9.9,-9.9,-9.9,-9.9,-9.9,-9.9,-9.9,-9.9,-9.9,0.059,0.208,0.309,0.382,0.516,0.629,0.738,0.809,0.703])

    tau1 = np.array([0.4010,0.3980,0.4020,0.4090,0.4450,0.5030,0.4740,0.4150,0.3540,0.3440,0.3500,0.3630,0.3810,0.4100,0.4570,0.4980,0.5250,0.5320,0.5370,0.5430,0.5320,0.5110,0.4870])
    tau2 = np.array([0.3460,0.3480,0.3450,0.3460,0.3640,0.4260,0.4660,0.4580,0.3880,0.3090,0.2660,0.2290,0.2100,0.2240,0.2660,0.2980,0.3150,0.3290,0.3440,0.3490,0.3350,0.2700,0.2390])
    phi1 = np.array([0.6440,0.6950,0.6980,0.7020,0.7210,0.7530,0.7450,0.7280,0.7200,0.7110,0.6980,0.6750,0.6430,0.6150,0.5810,0.5530,0.5320,0.5260,0.5340,0.5360,0.5280,0.5120,0.5100])
    phi2 = np.array([0.5520,0.4950,0.4990,0.5020,0.5140,0.5320,0.5420,0.5410,0.5370,0.5390,0.5470,0.5610,0.5800,0.5990,0.6220,0.6250,0.6190,0.6180,0.6190,0.6160,0.6220,0.6340,0.6040])
    dphiR= np.array([0.082,0.100,0.096,0.092,0.081,0.063,0.064,0.087,0.120,0.136,0.141,0.138,0.122,0.109,0.100,0.098,0.104,0.105,0.088,0.070,0.061,0.058,0.060])
    dphiV= np.array([0.080,0.070,0.070,0.030,0.029,0.030,0.022,0.014,0.015,0.045,0.055,0.050,0.049,0.060,0.070,0.020,0.010,0.008,0.000,0.000,0.000,0.000,0.000])
    R1   = np.array([105.000,110.000,111.670,113.100,112.130,97.930,85.990,79.590,81.330,90.910,97.040,103.150,106.020,105.540,108.390,116.390,125.380,130.370,130.360,129.490,130.220,130.720,130.000])
    R2   = np.array([272.000,270.000,270.000,270.000,270.000,270.000,270.040,270.090,270.160,270.000,269.450,268.590,266.540,265.000,266.510,270.000,262.410,240.140,195.000,199.450,230.000,250.390,210.000])

    #magnitude check
    if M <= mh[ip]:
        FE = (e0[ip]*U + e1[ip]*SS + e2[ip]*NS + e3[ip]*RS + e4[ip]*(M - mh[ip]) + e5[ip]*(M - mh[ip])**2)
    else:
        FE = (e0[ip]*U + e1[ip]*SS + e2[ip]*NS + e3[ip]*RS + e6[ip]*(M - mh[ip]))

    #Region check
    if region in (0,1):
        deltac3 = deltac3_gloCATW
    elif region == 3:
        deltac3 = deltac3_CHTU
    elif region in (2,4):
        deltac3 = deltac3_ITJA
    else:
        deltac3 = deltac3_gloCATW

    #Distance
    r = math.sqrt(Rjb**2 + h[ip]**2)
    FP = (c1[ip] + c2[ip]*(M - mref)) * math.log(r/rref) + (c3[ip] + deltac3[ip]) * (r - rref)

    PGA_ip = 1  # period index for PGA
    if not (abs(Vs30 - v_ref) < 1e-12 and ip == PGA_ip):
        PGA_r, _ = _BSSA_2014_sub(M, PGA_ip, Rjb, U, SS, NS, RS, region, z1, v_ref)
        if Vs30 <= vc[ip]:
            ln_Flin = c[ip] * math.log(Vs30 / v_ref)
        else:
            ln_Flin = c[ip] * math.log(vc[ip] / v_ref)
        f2 = f4[ip] * (math.exp(f5[ip]*(min(Vs30,760.0)-360.0)) - math.exp(f5[ip]*(760.0-360.0)))
        ln_Fnlin = f1 + f2 * math.log((PGA_r + f3)/f3)
        if z1 != 999:
            if region == 1:
                mu_z1 = math.exp((-7.15/4.0)*math.log((Vs30**4+570.94**4)/(1360.0**4+570.94**4)))/1000.0
            elif region == 2:
                mu_z1 = math.exp((-5.23/2.0)*math.log((Vs30**2+412.39**2)/(1360.0**2+412.39**2)))/1000.0
            else:
                mu_z1 = math.exp((-7.15/4.0)*math.log((Vs30**4+570.94**4)/(1360.0**4+570.94**4)))/1000.0
            dz1 = z1 - mu_z1
        else:
            dz1 = 0.0
        if z1 != 999:
            if period[ip] < 0.65:
                F_dz1 = 0.0
            elif (period[ip] >= 0.65) and (abs(dz1) <= (f7[ip]/f6[ip])):
                F_dz1 = f6[ip]*dz1
            else:
                F_dz1 = f7[ip]
        else:
            F_dz1 = 0.0
        FS = ln_Flin + ln_Fnlin + F_dz1
        lnY = FE + FP + FS
    else:
        lnY = FE + FP

    median = math.exp(lnY)

    if M <= 4.5:
        tau = tau1[ip];        phi_M = phi1[ip]
    elif 4.5 < M < 5.5:
        tau = tau1[ip] + (tau2[ip] - tau1[ip]) * (M - 4.5)
        phi_M = phi1[ip] + (phi2[ip] - phi1[ip]) * (M - 4.5)
    else:
        tau = tau2[ip];        phi_M = phi2[ip]

    if Rjb <= R1[ip]:
        phi_MR = phi_M
    elif R1[ip] < Rjb <= R2[ip]:
        phi_MR = phi_M + dphiR[ip] * (math.log(Rjb/R1[ip]) / math.log(R2[ip]/R1[ip]))
    else:
        phi_MR = phi_M + dphiR[ip]

    if Vs30 >= v2:
        phi_MRV = phi_MR
    elif v1 <= Vs30 <= v2:
        phi_MRV = phi_MR - dphiV[ip] * (math.log(v2/Vs30) / math.log(v2/v1))
    else:
        phi_MRV = phi_MR - dphiV[ip]

    sigma = math.sqrt(phi_MRV**2 + tau**2)

    #Output shaking intensities
    return median, sigma


################# PGA to RGB Mapping Function   #######################
# PGA (units of [g]) -> returns RGB in [0,1]
#Establish [g] boundaries for different colors
BASE_EDGES_G = np.array([0.0, 0.001, 0.0049, 0.010, 0.015, 0.0235, 0.030, 0.050, 0.0673,
                         0.100, 0.126, 0.200, 0.237, 0.350, 0.444, 0.833, 1.56, 1.75, np.inf], dtype=float)
#Establish base colors for given [g] boundaries
BASE_COLORS = np.array([
    [0.96, 0.96, 0.96],
    [0.90, 0.92, 1.00],
    [0.74, 0.82, 0.92],
    [0.68, 0.85, 0.90],
    [0.50, 1.00, 1.00],
    [0.56, 1.00, 0.83],
    [0.56, 1.00, 0.56],
    [0.80, 1.00, 0.18],
    [0.93, 1.00, 0.00],
    [1.00, 1.00, 0.00],
    [1.00, 0.90, 0.00],
    [1.00, 0.80, 0.00],
    [1.00, 0.70, 0.00],
    [1.00, 0.60, 0.00],
    [1.00, 0.47, 0.00],
    [1.00, 0.33, 0.00],
    [1.00, 0.22, 0.00],
    [1.00, 0.00, 0.00],
], dtype=float)

#Interpolation and color discretization functions
#Shaking intensity interpolation
def _geom_subedges(lo, hi, subdiv):
    if subdiv <= 1:
        return np.array([hi], dtype=float)
    if lo <= 0.0:
        return np.linspace(hi/subdiv, hi, subdiv)
    ratio = (hi/lo) ** (1.0/subdiv)
    return lo * (ratio ** np.arange(1, subdiv+1))
#Color discretization for smoother color gradient transitions on shake maps
def build_refined_palette_adaptive(base_edges, base_colors, subdiv_other=5, subdiv_low=14, low_cap=0.12, preserve_last=3):
    assert base_edges.ndim == 1 and base_colors.ndim == 2
    assert len(base_edges) == len(base_colors) + 1

    new_edges = [base_edges[0]]
    new_colors = []
    n_base = len(base_colors)
    cutoff = max(0, n_base - preserve_last)

    for i in range(n_base):
        lo, hi = base_edges[i], base_edges[i+1]
        c0 = base_colors[i]
        c1 = base_colors[min(i+1, n_base-1)]
        if i >= cutoff:
            new_edges.append(hi)
            new_colors.append(c0)
            continue
        subdiv = subdiv_low if hi <= low_cap else subdiv_other
        if subdiv <= 1:
            new_edges.append(hi)
            new_colors.append(c0)
            continue
        sub_edges = _geom_subedges(lo, hi, subdiv)
        t_vals = np.linspace(0.0, 1.0, subdiv+1)
        sub_colors = [(1-t)*c0 + t*c1 for t in t_vals[:-1]]
        for e, col in zip(sub_edges, sub_colors):
            new_edges.append(float(e))
            new_colors.append(np.asarray(col, dtype=float))
    return np.asarray(new_edges, dtype=float), np.vstack(new_colors).astype(float)

EDGES_G, COLORS = build_refined_palette_adaptive(BASE_EDGES_G, BASE_COLORS, subdiv_other=5, subdiv_low=14, low_cap=0.12, preserve_last=3)
# For vectorized [g] to RGB conversions
def pga_to_rgb_vec(pga_array):
    p = np.asarray(pga_array, dtype=float).ravel()
    rgb = np.full((p.size, 3), np.nan, dtype=float)
    good = np.isfinite(p)
    if not np.any(good):
        return rgb
    idx = np.digitize(p[good], EDGES_G) - 1
    idx = np.clip(idx, 0, len(COLORS)-1)
    # optional mid-range adjustment (as in your version)
    mid = (p[good] >= 0.0425) & (p[good] <= 0.444)
    idx[mid] = np.maximum(0, idx[mid] - 2)
    rgb[good] = COLORS[idx]
    return rgb  # in [0,1]

################### Polygon area nearest distance function ###################
def point_to_rotated_rect_distance(px, py, cx, cy, length_m, area_m2, theta, return_edge_distance=False):
    """
    Vectorized distance from points (px,py) to a rotated rectangle.
    length_m = rupture length, width = area_m2 / length_m.
    Returns 0 if inside (when return_edge_distance=False).

    return_edge_distance --> flag to return distance to edge for points inside rectangle. If false, return distance = 0 for points inside the area
    """
    #Create rectangle area
    eps = 1e-12
    L = max(float(length_m), eps)
    W = max(float(area_m2)/L, eps)
    hx, hy = 0.5*L, 0.5*W

    #Orient the rectangle in global vs. local coordinates
    x = px - cx
    y = py - cy
    c = np.cos(theta); s = np.sin(theta)
    x_local =  c*x + s*y
    y_local = -s*x + c*y

    dx = np.maximum(np.abs(x_local) - hx, 0.0)
    dy = np.maximum(np.abs(y_local) - hy, 0.0)
    outside_dist = np.hypot(dx, dy)

    #Check inside point edge distance flag
    if not return_edge_distance:
        return outside_dist

    #Calculate distance
    inside_mask = (np.abs(x_local) <= hx) & (np.abs(y_local) <= hy)
    edge_dist_inside = np.minimum(hx - np.abs(x_local), hy - np.abs(y_local))
    edge_dist_inside = np.maximum(edge_dist_inside, 0.0)

    return np.where(inside_mask, edge_dist_inside, outside_dist)

#################### Main Data Set Generator Function ###########################
# This function will implement the above functions to simulate ground motions and
# corresponding shakemaps for a given region. Earthquake rupture is randomized and
# site location is randomized (but constrained within California)
def simulate_shakemap_data(
    vs30_path,
    n_faults,
    grid_size=75,
    spacing_m=325.0,
    max_fault_length_m=380000.0,
    california_states_zip_url="https://www2.census.gov/geo/tiger/TIGER2025/STATE/tl_2025_us_state.zip",
    fix_M = None  # Can specify non-random magnitude to generate data for any given magnitude earthquake (e.g., 8.4 for testing)
):
    # Get CA boundary (EPSG:3310)
    state_zip = "us_states.zip"
    state_dir = "us_states"
    if not os.path.exists(state_zip):
        r = requests.get(california_states_zip_url)
        with open(state_zip, "wb") as f:
            f.write(r.content)
        with zipfile.ZipFile(state_zip, "r") as z:
            z.extractall(state_dir)

    states = gpd.read_file(f"{state_dir}/tl_2025_us_state.shp")
    california = states[states["NAME"] == "California"]
    california_proj = california.to_crs(epsg=3310)
    polygon_proj = california_proj.geometry.iloc[0]

    proj_to_geo = Transformer.from_crs("EPSG:3310", "EPSG:4326", always_xy=True)

    # Pull Vs30 data from moasic/raster
    if not os.path.exists(vs30_path):
        raise FileNotFoundError(f"{vs30_path} not found.")
    src = rasterio.open(vs30_path)
    raster_data = src.read(1)
    raster_transform = src.transform
    raster_nodata = src.nodata
    raster_crs = src.crs

    # Precompute grid offsets --> Discretizing map into 75 x 75 grid
    half = (grid_size - 1)/2.0
    offsets = np.linspace(-half*spacing_m, half*spacing_m, grid_size)  # meters
    ox, oy = np.meshgrid(offsets, offsets)  # shape (75,75)

    # Initialize output arrays
    layers_5 = np.zeros((n_faults, grid_size, grid_size, 5), dtype=np.float32)
    rgb_250  = np.zeros((n_faults, 250, 250, 3), dtype=np.float32)

    # Generate random seed
    rng = np.random.default_rng()

    count = 0
    attempts = 0
    max_attempts = 1_000_000

    #For each earthquake simulation, Loop over max_attampts till suitable rupture information is sampled
    while count < n_faults and attempts < max_attempts:
        attempts += 1

        # Random fault center inside CA (3310)
        minx, miny, maxx, maxy = polygon_proj.bounds
        fc_x = rng.uniform(minx, maxx)
        fc_y = rng.uniform(miny, maxy)
        if not polygon_proj.contains(Point(fc_x, fc_y)):
            continue

        # sample Magnitude in rnage of 3 to 8.5
        if fix_M is not None:
            M = float(fix_M)
        else:
            M = rng.uniform(3.0, 8.5)
        # Calculate rupture dimensions based on magnitude scaling relations
        #Length
        if M >= 5.2:
            rupture_length_km = 10 ** (-3.22 + 0.69 * M)
        else:
            rupture_length_km = rng.uniform(0.25, 2.3)
        rupture_length_m = rupture_length_km * 1000.0
        #Area
        if M >= 4.8 and M <= 7.9:
            rupture_area_km2 = 10 ** (-3.49 + 0.91 * M)
        elif M < 4.8:
            rupture_area_km2 = rng.uniform(0.75*M, 1.57*M)
        else:
            rupture_area_km2 = rng.uniform(5000, 7500)
        rupture_area_m2 = rupture_area_km2 * 1_000_000.0

        #Randomly orient the rupture in space
        theta = rng.uniform(0, 2*np.pi)
        dx = 0.5 * rupture_length_m * np.cos(theta)
        dy = 0.5 * rupture_length_m * np.sin(theta)
        p1 = Point(fc_x - dx, fc_y - dy)
        p2 = Point(fc_x + dx, fc_y + dy)

        # Find a random site center within CA and within max_fault_length_m to the rupture area
        center_ok = False
        for _ in range(500):
            #randomly site coordinates
            cx = rng.uniform(minx, maxx)
            cy = rng.uniform(miny, maxy)
            if not polygon_proj.contains(Point(cx, cy)):   #Check to ensure within California, if not loop again
                continue
            dist_to_area = point_to_rotated_rect_distance(
                np.array([cx]), np.array([cy]),
                fc_x, fc_y, rupture_length_m, rupture_area_m2, theta,
                return_edge_distance=False
            )[0]    #Calculate distance using initialzied function
            if dist_to_area <= max_fault_length_m:   #Check that distance to rupture is within the max allowed distance
                center_ok = True
                break
        if not center_ok:   #Reloop if center data not valid
            continue

        # Build and calculate distance to rupture for all points on the 75x75 grid
        px = cx + ox  # shape (75,75)
        py = cy + oy

        # Distance to rupture area (meters)
        R_vals = point_to_rotated_rect_distance(px, py, fc_x, fc_y, rupture_length_m, rupture_area_m2, theta, False)

        # Sample Vs30: convert (px,py) -> lon/lat
        lon_grid, lat_grid = proj_to_geo.transform(px, py)  # each (75,75)
        # Reproject lon/lat to the raster CRS if needed
        if raster_crs and raster_crs.to_string() != "EPSG:4326":
            xs, ys = rio_transform("EPSG:4326", raster_crs, lon_grid.ravel().tolist(), lat_grid.ravel().tolist())
        else:
            xs, ys = lon_grid.ravel(), lat_grid.ravel()

        rows_idx, cols_idx = rowcol(raster_transform, xs, ys, op=float)
        rows_idx = np.array(rows_idx, dtype=int)
        cols_idx = np.array(cols_idx, dtype=int)
        valid = (rows_idx >= 0) & (rows_idx < src.height) & (cols_idx >= 0) & (cols_idx < src.width)

        vs30_flat = np.full(rows_idx.shape, np.nan, dtype=float)
        if np.any(valid):
            sample_vals = raster_data[rows_idx[valid], cols_idx[valid]].astype(float)
            if raster_nodata is not None:
                sample_vals[sample_vals == raster_nodata] = np.nan
            vs30_flat[valid] = sample_vals

        Vs30_vals = vs30_flat.reshape(grid_size, grid_size)
        Vs30_vals[~np.isfinite(Vs30_vals) | (Vs30_vals <= 0)] = np.nan

        # PGA and RGB on 75x75
        R_km = R_vals / 1000.0
        pga = np.full((grid_size, grid_size), np.nan, dtype=float)
        good_mask = np.isfinite(Vs30_vals)
        if np.any(good_mask):
            med, _, _ = gmm_bssa2014(M, R_km[good_mask], Vs30_vals[good_mask], T=0.0, Fault_Type=0, region=1, z1=999)
            pga[good_mask] = med

        rgb75 = pga_to_rgb_vec(pga)  # returns (N,3)
        rgb75 = rgb75.reshape(grid_size, grid_size, 3)  # [0,1]
        # Keep north-up convention consistent
        rgb75_north_up = rgb75[::-1, :, :]

        # Fill 5-layer stack: [R, Vs30, Lat, Lon, M]
        layers_5[count, :, :, 0] = R_vals.astype(np.float32)
        layers_5[count, :, :, 1] = np.nan_to_num(Vs30_vals, nan=np.float32(np.nan)).astype(np.float32)
        layers_5[count, :, :, 2] = lat_grid.astype(np.float32)
        layers_5[count, :, :, 3] = lon_grid.astype(np.float32)
        layers_5[count, :, :, 4] = np.float32(M)  # broadcast

        # Store normalized RGB
        rgb_250[count] = rgb250_norm.astype(np.float32)

        count += 1

    if count < n_faults:   #If did not succcessfully generate enough valid data points
        print(f"Warning: only generated {count}/{n_faults} due to attempt limit.")

    return layers_5[:count], rgb_250[:count]


Generate Training Set using Data Set Generator Function


In [ ]:
rasterName = ""     #Name of the uploaded VS30 raster file that was downloaded from USGS
vs30_path = "/content/" + rasterName

X_train, Y_train = simulate_shakemap_data(
    vs30_path,
    n_faults=2400,
    fix_M=None     #Can apply a fixed magnitude
)

print("Training set shapes:")
print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)

# Save as .npy
np.save("X_train.npy", X_train)
np.save("Y_train.npy", Y_train)

Generate Validation Set using Data Set Generator Function


In [ ]:
X_dev, Y_dev = simulate_shakemap_data(
    vs30_path,
    n_faults=224,
    fix_M=None    #Can apply a fixed magnitude
)

print("Dev set shapes:")
print("X_dev:", X_dev.shape)
print("Y_dev:", Y_dev.shape)

# Save as .npy
np.save("X_dev.npy", X_dev)
np.save("Y_dev.npy", Y_dev)